In [1]:
# ==========================================
# STEP 1: IMPORTS & ENVIRONMENT SETUP
# ==========================================
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Conv2D, MaxPooling2D, Flatten, Input
import xgboost as xgb
from sklearn.model_selection import train_test_split

print("Libraries imported successfully. TensorFlow version:", tf.__version__)

Libraries imported successfully. TensorFlow version: 2.21.0


In [ ]:
import tensorflow as tf
import h5py
import pandas as pd
import numpy as np

tcir_path = 'TCIR-CPAC_IO_SH.h5'
print(f"Attempting to load TCIR Dataset from: {tcir_path}...")

try:
    # 'matrix' is a plain HDF5 dataset -> read with h5py
    with h5py.File(tcir_path, 'r') as f:
        X_tcir_images_raw = f['matrix'][()]

    # 'info' is stored as a pandas table (via HDFStore) -> read with pandas, not h5py
    info_df = pd.read_hdf(tcir_path, key='info')
    print("Available info columns:", info_df.columns.tolist())

    # Wind speed column in TCIR is typically named 'Vmax'
    y_tcir_winds_raw = info_df['Vmax'].values.astype(np.float32)

    print("TCIR Dataset loaded successfully!")
    print(f"Raw images shape: {X_tcir_images_raw.shape}")
    print(f"Raw wind speeds shape: {y_tcir_winds_raw.shape}")

    # 1. Handle NaNs in images
    X_tcir_images_raw = np.nan_to_num(X_tcir_images_raw, nan=0.0)

    # Handle NaNs in wind speeds
    if np.isnan(y_tcir_winds_raw).any():
        median_wind = np.nanmedian(y_tcir_winds_raw)
        y_tcir_winds_raw = np.nan_to_num(y_tcir_winds_raw, nan=median_wind)
        print(f"NaNs found in wind speeds, replaced with median: {median_wind:.2f}")

    # 2. Normalize pixel values
    #    TCIR channels have very different scales (IR ~ Kelvin, PMW, VIS, etc.)
    #    so per-channel normalization is safer than a single global max.
    X_tcir_images = X_tcir_images_raw.astype(np.float32)
    for c in range(X_tcir_images.shape[-1]):
        channel = X_tcir_images[..., c]
        c_min, c_max = np.nanmin(channel), np.nanmax(channel)
        if c_max > c_min:
            X_tcir_images[..., c] = (channel - c_min) / (c_max - c_min)
        else:
            X_tcir_images[..., c] = 0.0

    # 3. Resize to target size (TCIR native is 201x201x4)
    target_img_size = 64
    target_channels = 4

    print(f"Resizing images from {X_tcir_images.shape[1]}x{X_tcir_images.shape[2]} to {target_img_size}x{target_img_size}...")
    X_tcir_images_final = tf.image.resize(X_tcir_images, (target_img_size, target_img_size)).numpy()

    if X_tcir_images_final.shape[-1] != target_channels:
        print(f"Warning: channel count is {X_tcir_images_final.shape[-1]}, trimming to {target_channels}.")
        X_tcir_images_final = X_tcir_images_final[..., :target_channels]

    # Assign to globals for Module 2
    global X_img_real, y_img_real, real_num_samples, real_img_size, real_channels
    X_img_real = X_tcir_images_final
    y_img_real = y_tcir_winds_raw
    real_num_samples = X_img_real.shape[0]
    real_img_size = X_img_real.shape[1]
    real_channels = X_img_real.shape[-1]

    print(f"Processed TCIR images shape: {X_img_real.shape}")
    print(f"Processed TCIR wind speeds shape: {y_img_real.shape}")

except FileNotFoundError:
    print(f"Error: TCIR dataset not found at {tcir_path}.")
    # ... fallback code unchanged ...

except Exception as e:
    print(f"An error occurred while loading or processing the TCIR dataset: {e}")
    # ... fallback code unchanged ...

Attempting to load TCIR Dataset from: TCIR-CPAC_IO_SH.h5...
Available info columns: ['data_set', 'ID', 'lon', 'lat', 'time', 'Vmax', 'R35_4qAVG', 'MSLP']
TCIR Dataset loaded successfully!
Raw images shape: (23118, 201, 201, 4)
Raw wind speeds shape: (23118,)


In [ ]:
# ==========================================
# STEP 3: SYNTHETIC DATA & MODEL - MODULE 1
# ==========================================

# 1. Generate Synthetic Data (Shape: [samples, timesteps, features])
# Features: [latitude, longitude, wind_speed, pressure]
num_samples = 1000
timesteps = 8  # Past 24 hours (3-hourly intervals)
features = 4

X_track_synth = np.random.rand(num_samples, timesteps, features)
# Target: Predict next [lat, lon]
y_track_synth = np.random.rand(num_samples, 2)

# Train-Test Split
X_tr_train, X_tr_test, y_tr_train, y_tr_test = train_test_split(X_track_synth, y_track_synth, test_size=0.2)

# 2. Build LSTM Model
track_model = Sequential([
    Input(shape=(timesteps, features)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(16, activation='relu'),
    Dense(2, activation='linear') # Output next lat, lon
])

track_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 3. Train the Model
print("Training Module 1: Track Prediction (LSTM)...")
track_model.fit(X_tr_train, y_tr_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
# ==========================================
# STEP 4: REAL DATA & MODEL - MODULE 2 (Intensity Estimation)
# ==========================================

# Use the real TCIR data loaded in STEP 2
# X_img_real and y_img_real are assumed to be available from the previous cell
if 'X_img_real' not in globals() or 'y_img_real' not in globals():
    print("Error: Real TCIR data (X_img_real, y_img_real) not found. Please run STEP 2 and ensure data is loaded correctly.")
    print("Falling back to synthetic data for Module 2 to allow continued execution.")
    # Fallback to synthetic data if STEP 2 failed or was skipped
    num_samples = 1000
    img_size = 64
    X_data_for_module2 = np.random.rand(num_samples, img_size, img_size, 4).astype(np.float32)
    y_data_for_module2 = np.random.uniform(30, 150, num_samples).astype(np.float32)
    current_img_size = img_size
    current_channels = 4
else:
    print(f"Using real TCIR data for Module 2 with {X_img_real.shape[0]} samples.")
    X_data_for_module2 = X_img_real
    y_data_for_module2 = y_img_real
    current_img_size = real_img_size # Use the size determined during loading
    current_channels = real_channels # Use the channel count determined during loading

X_im_train, X_im_test, y_im_train, y_im_test = train_test_split(X_data_for_module2, y_data_for_module2, test_size=0.2, random_state=42)

# 2. Build CNN Model
intensity_model = Sequential([
    Input(shape=(current_img_size, current_img_size, current_channels)), # Dynamically set input shape
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='linear') # Output maximum sustained wind speed
])

intensity_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 3. Train the Model
print("Training Module 2: Intensity Estimation (CNN) with TCIR data...")
intensity_model.fit(X_im_train, y_im_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
# ==========================================
# STEP 5: RISK ZONE CLASSIFICATION - MODULE 3
# ==========================================

def classify_risk_zone(predicted_wind, distance_to_coast, vulnerability_index):
    """
    A multi-criteria classification mapping hazard and vulnerability to a zone.
    """
    risk_score = (predicted_wind * 0.5) - (distance_to_coast * 0.3) + (vulnerability_index * 0.2)

    if risk_score > 60:
        return "Red"
    elif risk_score > 30:
        return "Orange"
    else:
        return "Yellow"

# Generate synthetic district DataFrame
districts_df = pd.DataFrame({
    'District_ID': range(1, 101),
    'Distance_to_Path_km': np.random.uniform(10, 300, 100),
    'Predicted_Wind_kt': np.random.uniform(30, 150, 100),
    'Vulnerability_Index': np.random.uniform(10, 100, 100) # Combines exposure/elevation
})

# Apply zoning
districts_df['Risk_Zone'] = districts_df.apply(
    lambda row: classify_risk_zone(row['Predicted_Wind_kt'], row['Distance_to_Path_km'], row['Vulnerability_Index']), axis=1
)

print("\nModule 3 Classification Sample:")
print(districts_df[['District_ID', 'Predicted_Wind_kt', 'Risk_Zone']].head())

In [ ]:
# ==========================================
# STEP 6: SYNTHETIC DATA & MODEL - MODULE 4
# ==========================================

# 1. Prepare Features and Targets
# Features: Wind Speed, Distance, Vulnerability
X_dmg = districts_df[['Predicted_Wind_kt', 'Distance_to_Path_km', 'Vulnerability_Index']]
# Target: Synthetic Economic Damage (in millions USD/INR)
y_dmg = districts_df['Predicted_Wind_kt'] * districts_df['Vulnerability_Index'] * 0.1 + np.random.normal(0, 5, 100)

X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(X_dmg, y_dmg, test_size=0.2)

# 2. Build and Train XGBoost Regressor
dmg_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=50, max_depth=3)
dmg_model.fit(X_d_train, y_d_train)

# 3. Predict
districts_df['Estimated_Damage_M'] = dmg_model.predict(X_dmg)
print("\nModule 4 Damage Estimation Sample:")
print(districts_df[['District_ID', 'Risk_Zone', 'Estimated_Damage_M']].head())

In [ ]:
# ==========================================
# STEP 7: ALERT DISPATCH GENERATOR - MODULE 5
# ==========================================

def generate_cap_alert(district_data):
    alerts = []
    for _, row in district_data.iterrows():
        zone = row['Risk_Zone']
        if zone == "Red":
            urgency = "Immediate"
            instruction = "Evacuation advisory in effect. Activate emergency shelters immediately."
        elif zone == "Orange":
            urgency = "Expected"
            instruction = "Prepare to evacuate. Secure coastal properties and stage resources."
        else:
            urgency = "Future"
            instruction = "Monitor local weather stations for precautionary advisories."

        alert_json = {
            "identifier": f"ALERT-DIST-{int(row['District_ID'])}",
            "info": {
                "category": "Met",
                "event": "Tropical Cyclone",
                "urgency": urgency,
                "severity": zone,
                "description": f"Predicted wind speeds up to {row['Predicted_Wind_kt']:.1f} kt.",
                "instruction": instruction
            }
        }
        alerts.append(alert_json)
    return alerts

generated_alerts = generate_cap_alert(districts_df.head(3))
print("\nModule 5 Automated Alert Sample (CAP Format):")
print(json.dumps(generated_alerts, indent=2))

In [ ]:
# ==========================================
# STEP 8: MAKE DATASETS READY FOR DOWNLOAD
# ==========================================
# from google.colab import files # Commented out for Jupyter compatibility

print("Saving synthetic datasets locally...")

# Fallback for X_track_synth and y_track_synth if Module 1 was not executed or its state was lost
if 'X_track_synth' not in globals() or 'y_track_synth' not in globals():
    print("Warning: X_track_synth or y_track_synth not found. Generating minimal synthetic data for saving.")
    num_samples_fallback = 100 # Smaller fallback for track data
    timesteps_fallback = 8
    features_fallback = 4
    X_track_synth = np.random.rand(num_samples_fallback, timesteps_fallback, features_fallback).astype(np.float32)
    y_track_synth = np.random.rand(num_samples_fallback, 2).astype(np.float32)

# 1. Save Tabular Sequence Data
np.save('synthetic_track_features.npy', X_track_synth)
np.save('synthetic_track_targets.npy', y_track_synth)

# 2. Save Image Tensors (Simulating TCIR)
# These are now saved from the real data if loaded, otherwise synthetic.
np.save('synthetic_tcir_images.npy', X_img_real) # Use X_img_real which is either real or synthetic fallback
np.save('synthetic_tcir_winds.npy', y_img_real) # Use y_img_real which is either real or synthetic fallback

# Fallback for districts_df if Module 3 was not executed or its state was lost
if 'districts_df' not in globals():
    print("Warning: districts_df not found. Generating minimal synthetic DataFrame for saving.")
    districts_df = pd.DataFrame({
        'District_ID': range(1, 11), # Smaller fallback for districts
        'Distance_to_Path_km': np.random.uniform(10, 300, 10),
        'Predicted_Wind_kt': np.random.uniform(30, 150, 10),
        'Vulnerability_Index': np.random.uniform(10, 100, 10)
    })
    # Add dummy Risk_Zone if districts_df is regenerated here
    districts_df['Risk_Zone'] = np.random.choice(['Red', 'Orange', 'Yellow'], 10)

# 3. Save GIS/Zoning and Damage DataFrame
districts_df.to_csv('synthetic_district_risk.csv', index=False)

print("Files successfully written to local storage!")

# Uncomment below to trigger automated browser downloads:
# files.download('synthetic_district_risk.csv') # Commented out for Jupyter compatibility
# files.download('synthetic_tcir_images.npy') # Commented out for Jupyter compatibility

In [ ]:
print("Saving trained models locally...")

# Save Track Prediction (LSTM) Model
track_model.save('track_prediction_lstm_model.h5')

# Save Intensity Estimation (CNN) Model
intensity_model.save('intensity_estimation_cnn_model.h5')

# Save Damage Estimation (XGBoost) Model
dmg_model.save_model('damage_estimation_xgb_model.json')

print("Models successfully saved to Colab storage!")

In [ ]:
# Uncomment below to trigger automated browser downloads:
# files.download('track_prediction_lstm_model.h5') # Commented out for Jupyter compatibility
# files.download('intensity_estimation_cnn_model.h5') # Commented out for Jupyter compatibility
# files.download('damage_estimation_xgb_model.json') # Commented out for Jupyter compatibility